# Ford GoBike: trip duration and riding patterns
## Part I — Exploratory analysis
### Mohamed Feras Elghandour

**Question:** How do recorded trip duration and departure timing differ between subscriber and customer trips, and how do weekday/weekend context and station separation relate to those patterns?

**Source:** [Udacity-provided Ford GoBike February 2019 data](https://video.udacity-data.com/topher/2020/October/5f91cf38_201902-fordgobike-tripdata/201902-fordgobike-tripdata.csv). The raw file is supplied separately; see DATA.md. Each row is a trip, not a unique person. These are observational records for one month in the San Francisco Bay Area.

The main features are duration, user type, departure hour and day type. Station coordinates support a distance measure. Demographic fields are not needed for this question. No demographic inference or causal claim is made.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (10, 5), 'figure.dpi': 120,
                     'font.size': 11, 'axes.titlesize': 14})
USER_ORDER = ['Subscriber', 'Customer']
DAY_ORDER = ['Weekday', 'Weekend']
COLORS = {'Subscriber': '#0072B2', 'Customer': '#D55E00'}
MARKERS = {'Subscriber': 'o', 'Customer': 's'}

def finish(ax, title, xlabel, ylabel):
    """Apply consistent titles, units, and uncluttered chart formatting."""
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    sns.despine(ax=ax)
    ax.figure.tight_layout()

def log_minutes(ax):
    """Keep original minute labels on a logarithmic duration axis."""
    ax.set_yscale('log')
    ax.set_yticks([1, 2, 5, 10, 20, 60, 180, 600, 1440])
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_ylim(1, 1600)

def haversine_km(frame):
    """Calculate great-circle distance between recorded station coordinates."""
    lat1, lat2, lon1, lon2 = map(np.radians, [
        frame['start_station_latitude'], frame['end_station_latitude'],
        frame['start_station_longitude'], frame['end_station_longitude']])
    term = (np.sin((lat2 - lat1) / 2)**2
            + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2)**2)
    return 6371 * 2 * np.arcsin(np.sqrt(np.clip(term, 0, 1)))

trips = pd.read_csv(Path('201902-fordgobike-tripdata.csv'))
original_shape = trips.shape
trips['start_time'] = pd.to_datetime(trips['start_time'], errors='raise')
trips['end_time'] = pd.to_datetime(trips['end_time'], errors='raise')
trips['duration_min'] = trips['duration_sec'] / 60
trips['start_hour'] = trips['start_time'].dt.hour
trips['weekday'] = trips['start_time'].dt.dayofweek
trips['day_type'] = np.where(trips['weekday'] < 5, 'Weekday', 'Weekend')
trips['distance_km'] = haversine_km(trips)
trips['user_type'] = pd.Categorical(trips['user_type'], USER_ORDER)
trips['day_type'] = pd.Categorical(trips['day_type'], DAY_ORDER)
# Count calendar exposure, rather than treating the two day groups as equal.
calendar = trips[['start_time']].assign(date=trips['start_time'].dt.normalize())
calendar = calendar.drop_duplicates('date')
day_exposure = calendar['date'].dt.dayofweek.map(
    lambda day: 'Weekday' if day < 5 else 'Weekend').value_counts()

## Data structure and preparation
The source has 183,412 rows and 16 columns, with numeric duration and coordinates, categorical user type, station identifiers, and timestamps. Convert timestamps, express duration in minutes, and derive ordered hour/day groups. Great-circle station separation is an endpoint measure; it is **not actual riding distance or speed**. Retain all rows for summaries, including unusual long trips and identical endpoints. Only explicitly labeled chart windows/samples restrict plotted observations.

In [ ]:
print('Original shape:', original_shape)
print('Duplicate rows:', trips.duplicated().sum())
display(trips.isna().sum().rename('missing_values').to_frame())
display(trips[['duration_min', 'distance_km']].describe(percentiles=[.5, .95, .99]))
display(trips.groupby('user_type', observed=True)['duration_min'].agg(['count', 'median', 'mean']))
elapsed = (trips['end_time'] - trips['start_time']).dt.total_seconds()
print('Elapsed minus recorded seconds:', (elapsed - trips['duration_sec']).min(),
      (elapsed - trips['duration_sec']).max())
print('Calendar day counts:', day_exposure.to_dict())
print('Trips over 60 minutes:', (trips['duration_min'] > 60).sum())
print('Trips over 10 km endpoint separation:', (trips['distance_km'] > 10).sum())
print('Identical endpoint coordinates:', (trips['distance_km'] == 0).sum())

**Preparation observations:** No duplicate rows were found. Missing station IDs/names occur in 197 rows at each endpoint; birth year and gender have 8,265 missing values each. All fields used in the main analysis are complete, so dropping records for unrelated missing demographics would unnecessarily change the sample. Timestamp differences agree with recorded durations to within one second. February contains 20 weekdays and 8 weekend days. The data already have one trip per row; no pivot-to-long correction is required.

## Univariate exploration
Assume recorded timestamps and duration describe each trip consistently. First establish category sizes and distribution shapes; expect a right tail in duration but inspect it before choosing a scale.

### 1. Where do most trip durations lie?

In [ ]:
fig, ax = plt.subplots()
ax.hist(trips['duration_min'], bins=np.arange(0, 62, 2), color='#0072B2', edgecolor='white')
median_duration = trips['duration_min'].median()
ax.axvline(median_duration, color='#333333', linestyle='--', label=f'Median, all trips: {median_duration:.2f} min')
ax.set_xlim(0, 60)
ax.legend()
finish(ax, 'Most trips are short, with a right tail', 'Trip duration (minutes; 2-minute bins)', 'Number of trips')
plt.show()

**Observations:** The median is 8.57 minutes. The 0–60 minute window contains 99.07% of trips; 1,710 longer trips are outside the displayed bins, retained for all summaries. The right skew warrants a full-range logarithmic view.

### 2. What does the full duration tail look like?

In [ ]:
fig, ax = plt.subplots()
edges = np.geomspace(trips['duration_min'].min(), trips['duration_min'].max() * 1.001, 45)
ax.hist(trips['duration_min'], bins=edges, color='#0072B2', edgecolor='white')
ax.set_xscale('log')
ax.set_xticks([1, 2, 5, 10, 20, 60, 180, 600, 1440])
ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
finish(ax, 'The full distribution includes very long recorded trips', 'Trip duration (minutes; log-spaced bins)', 'Number of trips per log-spaced bin')
plt.show()

**Observations:** All 183,412 trips are included. Durations range from 1.02 to 1424.07 minutes. The mean (12.10) exceeds the median (8.57); log-spaced bins reveal the tail without treating unequal-width bins as a density.

### 3. Which trip category is larger?

In [ ]:
counts = trips['user_type'].value_counts().reindex(USER_ORDER)
fig, ax = plt.subplots()
bars = ax.bar(counts.index, counts.values, color=[COLORS[x] for x in USER_ORDER])
for bar, count in zip(bars, counts):
    ax.text(bar.get_x()+bar.get_width()/2, count + 2500, f'{count:,} ({count/len(trips):.1%})', ha='center')
ax.set_ylim(0, counts.max()*1.15)
finish(ax, 'Subscriber trips dominate the recorded volume', 'Recorded user type', 'Number of trips')
plt.show()

**Observations:** There are 163,544 subscriber trips (89.17%) and 19,868 customer trips (10.83%). Subsequent group profiles should use proportions when comparing shapes, because raw counts would mainly reflect this imbalance.

### 4. At what hours do trips start?

In [ ]:
counts = trips['start_hour'].value_counts().reindex(range(24), fill_value=0)
fig, ax = plt.subplots()
ax.bar(counts.index, counts, color='#0072B2')
ax.set_xticks(range(0, 24, 2))
finish(ax, 'Departure volume peaks in morning and late afternoon', 'Departure hour (recorded local clock, 0–23)', 'Number of trips')
plt.show()

**Observations:** The overall hourly distribution has peaks around 08:00 and 17:00. These are compatible with commute timing, but trip purpose is not recorded. The pooled shape may hide differences by day type and user type.

### 5. How far apart are recorded endpoints?

In [ ]:
fig, ax = plt.subplots()
ax.hist(trips['distance_km'], bins=np.arange(0, 10.25, .25), color='#0072B2', edgecolor='white')
ax.set_xlim(0, 10)
finish(ax, 'Most station pairs are separated by only a few kilometers', 'Great-circle endpoint separation (km; 0.25-km bins)', 'Number of trips')
plt.show()

**Observations:** Median endpoint separation is 1.43 km. There are 3,880 identical-coordinate trips and 11 trips above 10 km, outside this view. A zero separation can be a round trip and does not mean zero riding. The maximum (69.47 km) requires inspection rather than automatic deletion.

**Univariate reflection:** Duration is strongly skewed; use medians and logarithmic distribution displays. User types are imbalanced; compare normalized timing profiles. No arbitrary outlier removal is applied. Display limits must be disclosed.

## Bivariate exploration
Questions: Do customer trips last longer? Is endpoint separation associated with duration? Do user categories have different timing profiles? Assume trips are comparable as recorded, while recognizing repeated riders, station geography and the unequal calendar exposure.

### 6. Does station separation explain trip duration?

In [ ]:
window = trips.loc[(trips['distance_km'] <= 10) & (trips['duration_min'] <= 60)]
sample = window.sample(n=min(5000, len(window)), random_state=42)
fig, ax = plt.subplots()
ax.scatter(sample['distance_km'], sample['duration_min'], s=10, alpha=.18, color='#0072B2', rasterized=True)
finish(ax, 'Longer endpoint separation has substantial duration variation', 'Great-circle endpoint separation (km)', 'Trip duration (minutes)')
print(f'Display window: {len(window):,}/{len(trips):,} trips; random sample: {len(sample):,}')
print('Full-data Pearson correlation:', trips[['distance_km', 'duration_min']].corr().iloc[0, 1])
print('Full-data Spearman correlation:', trips[['distance_km', 'duration_min']].corr(method='spearman').iloc[0, 1])
plt.show()

**Observations:** The 5,000-trip reproducible sample avoids overplotting. Only trips within 10 km and 60 minutes are sampled; the printed coverage makes the restriction explicit. Full-data Pearson correlation is about 0.127, while Spearman rank correlation is about 0.728. This indicates a strong monotonic relationship that the skewed tail and unusual endpoints obscure in a linear correlation. Wide duration variation still remains at similar distances; station separation alone does not determine travel time and cannot measure actual speed.

### 7. How do duration distributions differ by user type?

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=trips, x='user_type', y='duration_min', order=USER_ORDER,
            color='#8AB8CF', showfliers=True, flierprops={'markersize': 1, 'alpha': .15}, ax=ax)
log_minutes(ax)
finish(ax, 'Customer trips have a higher median duration', 'Recorded user type', 'Trip duration (minutes, logarithmic scale)')
plt.show()

**Observations:** Median duration is 8.17 minutes for subscribers and 13.20 minutes for customers. All trips contribute to box statistics and points. Both distributions have long right tails; group membership does not explain every trip.

### 8. How do hourly volumes vary across the week?

In [ ]:
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
day_counts = calendar['date'].dt.dayofweek.value_counts().reindex(range(7))
hour_table = pd.crosstab(trips['weekday'], trips['start_hour']).reindex(index=range(7), columns=range(24), fill_value=0)
hour_rate = hour_table.div(day_counts, axis=0)
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(hour_rate, cmap='Blues', yticklabels=day_names, ax=ax,
            cbar_kws={'label': 'Average trips per occurrence of this weekday'})
finish(ax, 'Weekday departure peaks contrast with broader weekend activity', 'Departure hour (0–23)', 'Day of week')
plt.show()

**Observations:** Weekdays show sharper morning and evening peaks; weekends shift toward daytime. Each row is divided by its number of calendar occurrences, so cells compare average daily volumes rather than unequal monthly exposure. All seven weekdays occur four times in February 2019.

### 9. Do the two user groups have different hourly profiles?

In [ ]:
hour_user = pd.crosstab(trips['start_hour'], trips['user_type']).reindex(index=range(24), columns=USER_ORDER, fill_value=0)
hour_share = hour_user.div(hour_user.sum(axis=0), axis=1) * 100
fig, ax = plt.subplots()
for user in USER_ORDER:
    ax.plot(hour_share.index, hour_share[user], color=COLORS[user], marker=MARKERS[user], markersize=4, label=user)
ax.set_xticks(range(0, 24, 2))
ax.legend(title='Recorded user type')
finish(ax, 'Both user groups show morning and evening departure peaks', 'Departure hour (0–23)', 'Share of each user type’s monthly trips (%)')
plt.show()

**Observations:** Both groups peak at 17:00 and show morning demand near 08:00. Customers have relatively more afternoon departures. Normalization removes the category-size imbalance; it does not identify individual rider behavior or prove commute/leisure purpose.

### 10. How does average daily trip volume differ by day type?

In [ ]:
day_user = pd.crosstab(trips['day_type'], trips['user_type']).reindex(index=DAY_ORDER, columns=USER_ORDER)
daily_rate = day_user.div(day_exposure.reindex(DAY_ORDER), axis=0)
fig, ax = plt.subplots()
daily_rate.plot.bar(ax=ax, color=[COLORS[x] for x in USER_ORDER], rot=0)
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3)
ax.set_ylim(0, daily_rate.to_numpy().max()*1.18)
ax.legend(title='Recorded user type')
finish(ax, 'Subscriber daily volume drops more on weekends', 'Day type', 'Average trips per calendar day')
display(daily_rate.round(1))
plt.show()

**Observations:** Customer weekend trips account for 28.36% of customer trips, versus 15.47% for subscribers. There are 20 weekdays and 8 weekend days, so the clustered bars divide monthly totals by those exposures. Customer daily demand changes less across the two day types than subscriber demand.

### 11. Are customer endpoints farther apart?

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(data=trips, x='user_type', y='distance_km', order=USER_ORDER, color='#8AB8CF', showfliers=False, ax=ax)
finish(ax, 'Endpoint separation is similar across user types', 'Recorded user type', 'Great-circle endpoint separation (km)')
print('Outlier points hidden; all trips included in quartiles.')
display(trips.groupby('user_type', observed=True)['distance_km'].agg(['median', 'mean']))
plt.show()

**Observations:** Median endpoint separations are 1.41 km for subscribers and 1.68 km for customers. The separation difference is much smaller than the duration difference. This weakens a simple explanation that customers always last longer because their endpoints are much farther apart. Outlier symbols are hidden for readability, but statistics use all trips.

**Bivariate reflection:** The user-type duration difference is clear, but endpoint separation does not fully account for duration variation. Daily exposure and category size affect raw counts, so use average daily volume and within-group hourly percentages. Next inspect whether day type modifies the duration and timing differences.

## Multivariate exploration
Assumption: calendar day type provides useful context but is not a trip-purpose label. Compare user type, duration and departure hour within day type. Keep category order, scales and colors consistent. Small hour-level groups and long tails can make means unstable; use descriptive medians without causal interpretation.

### 12. Does day type modify each user group’s hourly profile?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
for ax, day_type in zip(axes, DAY_ORDER):
    subset = trips.loc[trips['day_type'] == day_type]
    table = pd.crosstab(subset['start_hour'], subset['user_type']).reindex(index=range(24), columns=USER_ORDER, fill_value=0)
    shares = table.div(table.sum(axis=0), axis=1) * 100
    for user in USER_ORDER:
        ax.plot(shares.index, shares[user], color=COLORS[user], marker=MARKERS[user],
                markersize=4, linestyle='-' if user == 'Subscriber' else '--', label=user)
    ax.set(title=day_type, xlabel='Departure hour (0–23)', xticks=range(0, 24, 3), ylim=(0, 15))
axes[0].set_ylabel('Share of user/day-type trips (%)')
axes[1].legend(title='Recorded user type')
fig.suptitle('Day type changes departure timing in both user groups', fontsize=15)
sns.despine(fig=fig)
fig.tight_layout()
plt.show()

**Observations:** Weekday morning and evening peaks appear in both groups. Weekend profiles are broader and concentrated later in the day. Each line sums to 100% within its user/day-type group; matching panel scales permit comparison of shapes, not total trip volumes. This qualifies the pooled hourly story.

### 13. Is the duration gap present on both weekdays and weekends?

In [ ]:
grid = sns.catplot(data=trips, x='user_type', y='duration_min', col='day_type',
                   order=USER_ORDER, col_order=DAY_ORDER, kind='box', showfliers=False,
                   color='#8AB8CF', height=4.5, aspect=1.25, sharey=True)
for ax in grid.axes.flat:
    ax.set_yscale('log')
    ax.set_ylim(1, 100)
    ax.set_yticks([1, 2, 5, 10, 20, 60, 100])
    ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
grid.set_axis_labels('Recorded user type', 'Trip duration (minutes, log scale)')
grid.set_titles('{col_name}')
grid.figure.suptitle('The customer duration gap persists within both day types', y=1.04)
display(trips.groupby(['user_type', 'day_type'], observed=True)['duration_min'].agg(['count', 'median']))
plt.show()

**Observations:** Customer medians are 12.40 minutes on weekdays and 16.05 on weekends. Subscriber medians are 8.18 and 8.13. Thus the pooled gap persists within each day type, and customer median duration rises more on weekends. Boxes use all observations; outlier symbols are hidden and display limits are 1–100 minutes.

### 14. How do median durations vary by hour, user type and day type?

In [ ]:
hour_summary = trips.groupby(['day_type', 'user_type', 'start_hour'], observed=True)['duration_min'].agg(['median', 'count']).reset_index()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True, sharey=True)
for ax, day_type in zip(axes, DAY_ORDER):
    for user in USER_ORDER:
        rows = hour_summary.loc[(hour_summary['day_type'] == day_type) & (hour_summary['user_type'] == user)]
        ax.plot(rows['start_hour'], rows['median'], color=COLORS[user], marker=MARKERS[user],
                linestyle='-' if user == 'Subscriber' else '--', markersize=4, label=user)
    ax.set(title=day_type, xlabel='Departure hour (0–23)', xticks=range(0, 24, 3))
axes[0].set_ylabel('Median trip duration (minutes)')
axes[1].legend(title='Recorded user type')
fig.suptitle('Duration differences vary with departure context', fontsize=15)
sns.despine(fig=fig)
fig.tight_layout()
print('Smallest hour/user/day group:', hour_summary['count'].min())
plt.show()

**Observations:** Customers generally have longer median trips throughout the day, particularly on weekend afternoons. The smallest hour/user/day group contains only five trips, so isolated early-hour fluctuations should not be read as a robust trend. These are exact descriptive group medians for this month, not estimates of independent person-level behavior.

### 15. Does the duration-distance relationship look similar across contexts?

In [ ]:
window = trips.loc[(trips['distance_km'] <= 10) & (trips['duration_min'] <= 60)]
sample = window.sample(n=min(3000, len(window)), random_state=17)
fig, ax = plt.subplots(figsize=(11, 5.5))
sns.scatterplot(data=sample, x='distance_km', y='duration_min', hue='user_type',
                hue_order=USER_ORDER, palette=COLORS, style='day_type', style_order=DAY_ORDER,
                markers={'Weekday': 'o', 'Weekend': 'X'}, s=22, alpha=.35, ax=ax)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='User and day type')
finish(ax, 'User and day context coexist with wide duration variation', 'Great-circle endpoint separation (km)', 'Trip duration (minutes)')
print(f'Display window: {len(window):,}/{len(trips):,}; reproducible random sample: {len(sample):,}')
plt.show()

**Observations:** Colors encode user type and shapes encode day type. Wide vertical variation remains at comparable endpoint separations; no clean separation suggests that distance or day type alone determines duration. The sample preserves the natural imbalance and is not suitable for comparing group counts. The same 10-km/60-minute restriction as chart 6 applies.

## Unusual endpoints: a targeted check
Question: Are the largest station separations plausible or necessarily bad records? Inspect their station names and durations before deciding whether they should be removed.

In [ ]:
display(trips.nlargest(5, 'distance_km')[['start_station_name', 'end_station_name', 'distance_km', 'duration_min']])

**Observations:** The largest endpoint separation is about 69.47 km, from Montgomery St BART to Palm St at Willow St, with about 115.75 recorded minutes. Other large separations cross the bay. These could reflect unusual travel, multimodal movement, or recording issues; the file alone cannot resolve that. Retain them, disclose them, and avoid interpreting endpoint distance as a route or speed. The main user-duration and timing findings do not depend on deleting these records.

## Conclusions and reflection
Subscriber trips account for 89.17% of volume. Customer trips have a 13.20-minute median versus 8.17 minutes for subscribers. The duration gap remains within both day types; customers rise from 12.40 minutes on weekdays to 16.05 on weekends while subscribers remain near 8.1 minutes. Both categories show weekday commute-hour peaks; weekend departures are broader and later. This supports planning around weekday peaks while considering different duration patterns across user and day groups.

Endpoint separation has a strong monotonic association with duration, but durations still vary widely at similar separations and similar endpoints do not imply similar routes. The exploration moved from pooled counts to normalized timing profiles, per-calendar-day volumes and subgroup medians to address imbalance and exposure. Full-range logarithmic duration views complement labeled linear windows.

Limitations: one month, one bike-share system, trips rather than unique riders, no observed trip purpose, unknown station-record anomalies, and no causal identification. These results suggest operational questions; they do not establish why customers ride longer, predict other seasons, or justify a causal pricing change. Future work should compare multiple months, validate anomalous station pairs and examine station-level demand before making specific capacity decisions.